In [1]:
import os
import GEOparse
import pandas as pd


# Get GEO_ID from parent folder name
GEO_ID = os.path.basename(os.path.dirname(os.path.abspath('__file__')))

gse = GEOparse.get_GEO(geo=GEO_ID, silent=True)

# map sample title to metadata
metadata = {}

for key, value in gse.gsms.items():
    sample_title = value.metadata['title'][0]

    metadata_object = {"sample_title": sample_title}
    for item in value.metadata['characteristics_ch1']:
        if 'survival: ' in item:
            item = item.replace('survival: ', '')

        k, v = item.split(':')
        metadata_object[k.strip()] = v.strip()

    metadata[key] = metadata_object


# Create metadata DataFrame and join as columns to df_genes (rows=samples)
metadata_df = pd.DataFrame.from_dict(metadata, orient='index')
metadata_df.index.name = 'samples'
metadata_df.head()

,sample_title,tissue,age,Stage,response_to_platinum_therapy,recurrence,progression-free-survival_days,live/dead,overall-survival_days
samples,,,,,,,,,
GSM5050316,hgsc_01,high grade serous ovarian carcinoma,45,III,No recurrence,0,3756,live,3756
GSM5050317,hgsc_02,high grade serous ovarian carcinoma,75,III,Not available,Not available,Not available,live,279
GSM5050318,hgsc_03,high grade serous ovarian carcinoma,37,III,No recurrence,0,3050,live,3050
GSM5050319,hgsc_04,high grade serous ovarian carcinoma,43,III,Not available,Not available,Not available,live,215
GSM5050320,hgsc_05,high grade serous ovarian carcinoma,77,III,No recurrence,0,1084,live,1084


# Use NCBI provided expression data

In [2]:
# Raw counts
ncbi_data_url = f"https://www.ncbi.nlm.nih.gov/geo/download/?type=rnaseq_counts&acc={GEO_ID}&format=file&file={GEO_ID}_raw_counts_GRCh38.p13_NCBI.tsv.gz"

df_raw_counts: pd.DataFrame = pd.read_csv(ncbi_data_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None

url = "https://www.ncbi.nlm.nih.gov/geo/download/?format=file&type=rnaseq_counts&file=Human.GRCh38.p13.annot.tsv.gz"
df_gene_annotations = pd.read_csv(
    url,
    sep="\t",
    compression="gzip"
)
gene_mapping = (
    df_gene_annotations[["GeneID", "Symbol"]]
    .dropna(subset=["GeneID", "Symbol"])
    .set_index("GeneID")["Symbol"]
    .to_dict()
)
# Keep only columns that have a mapping
mappable = [c for c in df_raw_counts.columns if c in gene_mapping]
df_raw_counts = df_raw_counts[mappable].rename(columns=gene_mapping)
# df_raw_counts = df_raw_counts.rename(columns=gene_mapping)

data_file_name = f'{GEO_ID}_raw_counts_NCBI.csv'
df_combined = metadata_df.join(df_raw_counts, how='inner')

df_combined.reset_index()
df_combined.to_csv(data_file_name, index=False)

df_combined.head()

/var/folders/h1/fnfx16m50jvd8ddpcs93g9mw0000gn/T/ipykernel_99127/3990720147.py:10: DtypeWarning: Columns (8,9) have mixed types. Specify dtype option on import or set low_memory=False.
  df_gene_annotations = pd.read_csv(


,sample_title,tissue,age,Stage,response_to_platinum_therapy,recurrence,progression-free-survival_days,live/dead,overall-survival_days,DDX11L1,...,ND4,TRNH,TRNS2,TRNL2,ND5,ND6,TRNE,CYTB,TRNT,TRNP
samples,,,,,,,,,,,,,,,,,,,,,
GSM5050316,hgsc_01,high grade serous ovarian carcinoma,45,III,No recurrence,0,3756,live,3756,1,...,297842,985,1162,1082,124388,32428,2535,136526,346,4023
GSM5050317,hgsc_02,high grade serous ovarian carcinoma,75,III,Not available,Not available,Not available,live,279,4,...,259319,95,103,89,59928,16492,1256,100318,61,2739
GSM5050319,hgsc_04,high grade serous ovarian carcinoma,43,III,Not available,Not available,Not available,live,215,8,...,358318,115,129,131,90898,42200,3638,127355,117,3072
GSM5050320,hgsc_05,high grade serous ovarian carcinoma,77,III,No recurrence,0,1084,live,1084,4,...,437784,432,572,535,150110,38605,3175,159766,351,4818
GSM5050322,hgsc_08,high grade serous ovarian carcinoma,55,III,No recurrence,0,1113,live,1113,5,...,481399,35,37,47,157046,41704,2310,120425,216,1672


# Generate description

In [3]:
import json
import os
import sys
from pathlib import Path

project_root = Path().resolve().parent.parent  # adjust based on notebook depth
if str(project_root) not in sys.path:
    sys.path.insert(0, str(project_root))

from geo.scripts.generate_descriptions import generate_description_from_geo

description_file_path = 'description.json'


if not os.path.exists(description_file_path):
    with open(description_file_path, 'w') as f:
        json.dump({}, f)

with open(description_file_path, 'r') as f:
    desc_data = json.load(f)

if not desc_data:
    desc_data = generate_description_from_geo(GEO_ID)
    print("Description generated")

desc_data['data_summary'] = {
    'samples': metadata_df.shape[0],
    'clinical_features': metadata_df.shape[1],
    'genes': df_raw_counts.shape[1]
}
print("Description updated")

desc_data['data_file_names'] = [data_file_name]

with open(description_file_path, "w") as f:
    json.dump(desc_data, f, indent=4)

Description updated


# Use author provided expression data


In [4]:
metadata_df = metadata_df.reset_index()
metadata_df = metadata_df.set_index('sample_title')

tpm_url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE165808&format=file&file=GSE165808%5Fhgsc%5Ftpm%2Etsv%2Egz"
df_tpm: pd.DataFrame = pd.read_csv(tpm_url, sep='\t', index_col=0, compression='gzip')
df_tpm = df_tpm.T
df_tpm.index.name = "samples"
df_tpm.columns.name = None

raw_counts_url = "https://www.ncbi.nlm.nih.gov/geo/download/?acc=GSE165808&format=file&file=GSE165808%5Fhgsc%5Fraw%5Fcount%2Etsv%2Egz"
df_raw_counts: pd.DataFrame = pd.read_csv(raw_counts_url, sep='\t', index_col=0, compression='gzip')
df_raw_counts = df_raw_counts.T
df_raw_counts.index.name = "samples"
df_raw_counts.columns.name = None


df_tpm = metadata_df.join(df_tpm, how='inner')
df_tpm.reset_index(inplace=True)
# INFO: Uncomment to save the data to disk
# df_tpm.to_csv(f'{GEO_ID}_TPM.csv', index=False)

df_raw_counts = metadata_df.join(df_raw_counts, how='inner')
df_raw_counts.reset_index(inplace=True)
# INFO: Uncomment to save the data to disk
# df_raw_counts.to_csv(f'{GEO_ID}_RAW_COUNTS.csv', index=False)

df_tpm.head()

,sample_title,samples,tissue,age,Stage,response_to_platinum_therapy,recurrence,progression-free-survival_days,live/dead,overall-survival_days,...,ZWILCH|55055,ZWINT|11130,ZXDA|7789,ZXDB|158586,ZXDC|79364,ZYG11A|440590,ZYG11B|79699,ZYX|7791,ZZEF1|23140,ZZZ3|26009
0,hgsc_01,GSM5050316,high grade serous ovarian carcinoma,45,III,No recurrence,0,3756,live,3756,...,9.935,94.050,2.530,6.417,19.264,0.772,10.060,211.415,10.265,32.151
1,hgsc_02,GSM5050317,high grade serous ovarian carcinoma,75,III,Not available,Not available,Not available,live,279,...,20.015,137.180,5.304,10.049,24.274,0.000,9.037,139.444,9.354,42.972
2,hgsc_03,GSM5050318,high grade serous ovarian carcinoma,37,III,No recurrence,0,3050,live,3050,...,17.867,54.040,14.661,18.728,34.367,0.492,22.414,120.603,18.284,48.803
3,hgsc_04,GSM5050319,high grade serous ovarian carcinoma,43,III,Not available,Not available,Not available,live,215,...,10.638,40.354,1.952,4.714,26.546,0.337,11.456,177.172,6.028,19.448
4,hgsc_05,GSM5050320,high grade serous ovarian carcinoma,77,III,No recurrence,0,1084,live,1084,...,22.381,250.393,2.065,8.062,20.418,0.261,11.950,58.286,5.225,32.299
